# Phase 4: Production ML Deployment & MLOps Infrastructure

Welcome to the final phase of the AuraCart MLOps initiative. In this notebook, we bridge the gap between offline experimentation and production inference. We will programmatically interact with Google Cloud Platform (GCP) to register our champion model, containerize it using Vertex AI's pre-built environments, and instantiate a live RESTful API endpoint.

As mandated by the AuraCart mandate, this system provides a unified, scalable interface for frontend microservices to query predictive logic in real-time using JSON payloads.

In [ ]:
import os
import joblib
from google.cloud import storage
from google.cloud import aiplatform

# --- Configuration Variables ---
PROJECT_ID = "your-project-id" # PLACEHOLDER: Replace with your actual GCP Project ID
REGION = "us-central1"
BUCKET_NAME = f"{PROJECT_ID}-auracart-ml-artifacts"
MODEL_NAME = "customer-segmentation-champion"
ENDPOINT_NAME = "auracart-prediction-endpoint"
MODEL_ARTIFACT_PATH = "../artifacts/model.joblib"
REQUIREMENTS_FILE = "../artifacts/requirements.txt"

print(f"Deployment system initialized targeting Project: {PROJECT_ID}")

### Environment Requirements Generation

To ensure the Vertex AI pre-built container has the correct dependencies (Scikit-Learn, Pandas, Imbalanced-Learn), we generate a `requirements.txt` file as an deployment artifact. This file will be uploaded to Cloud Storage alongside the model binary.

In [ ]:
requirements_content = """
scikit-learn==1.3.0
pandas==2.0.3
numpy==1.24.3
joblib==1.3.1
imbalanced-learn==0.11.0
"""

with open(REQUIREMENTS_FILE, "w") as f:
    f.write(requirements_content.strip())

print(f"Deployment requirements artifact generated at: {REQUIREMENTS_FILE}")

### Task 4.2.5: Uploading Artifacts to Google Cloud Storage (GCS)

Vertex AI requires model artifacts to reside in a GCS bucket. We programmatically create a bucket and upload our `model.joblib` and `requirements.txt` for registry ingestion.

In [ ]:
print("Initializing Cloud Storage Client...")
storage_client = storage.Client(project=PROJECT_ID)

# Create Bucket if it doesn't exist
bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Bucket {BUCKET_NAME} created.")

# Upload Model Artifact
model_blob = bucket.blob("model/model.joblib")
model_blob.upload_from_filename(MODEL_ARTIFACT_PATH)
print("Model artifact uploaded to GCS.")

# Upload Requirements Artifact
req_blob = bucket.blob("model/requirements.txt")
req_blob.upload_from_filename(REQUIREMENTS_FILE)
print("Dependencies artifact uploaded to GCS.")

### Task 4.3: Vertex AI Model Registry & Endpoint Deployment

We now register our model with Vertex AI, pointing it to the GCS directory and selecting a pre-built Scikit-Learn prediction container. Once registered, we deploy this model to a dedicated managed endpoint for live inference.

In [ ]:
print("Initializing Vertex AI SDK...")
aiplatform.init(project=PROJECT_ID, location=REGION)

# 1. Upload Model to Registry
print("Registering model in Vertex AI Registry...")
model_registered = aiplatform.Model.upload(
    display_name=MODEL_NAME,
    artifact_uri=f"gs://{BUCKET_NAME}/model/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest",
)

# 2. Deploy to Endpoint
print("Instantiating Live Prediction Endpoint (this may take several minutes)... ")
endpoint = model_registered.deploy(
    endpoint_display_name=ENDPOINT_NAME,
    machine_type="n1-standard-2", # Minimal compute footprint for operational efficiency
)

print(f"Successfully deployed AuraCart Prediction Engine to Endpoint ID: {endpoint.resource_name}")

### Task 4.3.4: Verifying the Live Prediction Pipeline

We simulate a real-world transaction from the AuraCart frontend. We send a JSON payload representing a new e-commerce event to the deployed Vertex AI endpoint and observe the classification result.

In [ ]:
import pandas as pd

# Simulate input features for a New Transaction using all 21 input features
test_data = {
    'price': [150.0], 'quantity': [2], 'shipping_delay_days': [4], 'product_popularity': [15],
    'order_month': [4], 'order_day': [15], 'order_dayofweek': [2], 'order_hour': [14],
    'shipping_month': [4], 'shipping_day': [19], 'shipping_dayofweek': [6], 'shipping_hour': [10],
    'category': ['Electronics'], 'payment_method': ['Credit Card'], 'device_type': ['Mobile'],
    'channel': ['Direct'], 'shipping_city': ['Austin'], 'shipping_state': ['TX'],
    'billing_city': ['Austin'], 'billing_state': ['TX'], 'is_weekend_order': [0]
}

test_instance_df = pd.DataFrame(test_data)

print("Querying endpoint with the enhanced 21-feature schema...")
# Note: In a real Vertex query, this df would be converted to a list for the 'instances' key
prediction = endpoint.predict(instances=test_instance_df.values.tolist())

print(f"Prediction Response Structure: {prediction}")
print(f"Assigned Customer Segment: {prediction.predictions[0]}")